# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, using their `@id` identifiers.

In [ ]:
# List all record sets by their @id and name/description
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in metadata.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"- @id: {rs.id}\n  name: {rs.name}\n  description: {rs.description if hasattr(rs,'description') else ''}\n")
    # For demonstration, print the fields for the first record set
    record_set = record_sets[0]
    print(f"Fields in record set '@id: {record_set.id}':")
    for field in record_set.fields:
        print(f"  - Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets defined in the metadata
dataframes = {}
record_set_ids = [rs.id for rs in (metadata.record_sets or [])]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet: {record_set_id}; Columns: {df.columns.tolist()}")
        print(f"Sample data from record set '{record_set_id}':")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
Select a numeric field and a group field from the data loaded above (referenced by their `@id`).

In [ ]:
# Set up for EDA: pick the first record set (if available)
if dataframes:
    # Try to pick the first available dataframe
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # List numeric fields
    numeric_candidates = df.select_dtypes(include=[float, int]).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Use first numeric field
    else:
        numeric_field_id = df.columns[0]  # fallback
    print(f"Using field '{numeric_field_id}' for numeric analysis.")

    # Filtering: set a threshold for numeric field
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else df
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

    # Grouping
    # Try to pick a group field that is not the numeric field, and looks categorical
    group_field_candidates = [c for c in df.columns if c != numeric_field_id and df[c].nunique() < (0.5*len(df))]
    if group_field_candidates:
        group_field_id = group_field_candidates[0]
        print(f"Grouping by field '{group_field_id}'")
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No DataFrames loaded; cannot perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In this example, if a numeric field is available, its distribution and relationship to a group field are plotted.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    if group_field_candidates:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and process a Croissant-based dataset using `mlcroissant` and pandas. Key findings, distributions, and group differences can be further analyzed based on this workflow.

You can extend this notebook with domain-specific analysis or additional processing as appropriate for your scientific or analytic goals.